# Automated Road Damage Detection & Distress Assessment (YOLOv8)
### Conference Research & Experimental Training Pipeline

This notebook trains and benchmarks an automated computer vision pipeline for **Road Damage Detection and Classification** on real-world road inspection imagery.

**Damage Categories (5 Classes):**
1. `Longitudinal Crack` (Parallel to road direction)
2. `Transverse Crack` (Perpendicular to centerline)
3. `Alligator Crack` (Interconnected fatigue fissures)
4. `Other Corruption` (General asphalt wear and spalling)
5. `Pothole` (High-hazard road depressions)

**Supports Dual-Modal Input:** Single Imagery & Continuous Dashcam/Survey Video Streams.

In [ ]:
# STEP 1: Environment Setup & GPU Verification
%pip install -q ultralytics pyyaml opencv-python-headless

import os
import yaml
import torch
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Accelerating on:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU (Enable GPU in Kaggle Notebook Settings for fast training).")

In [ ]:
# STEP 2: Dataset Discovery & YOLO YAML Configuration
input_dir = Path("/kaggle/input")
candidates = list(input_dir.glob("**/randomized_dataset"))

if candidates:
    dataset_root = candidates[0]
    print("Found dataset at:", dataset_root)
else:
    # Fallback to current working directory if run locally
    dataset_root = Path("./dataset")
    print("Dataset path:", dataset_root)

config = {
    "path": str(dataset_root),
    "train": "train/images",
    "val": "val/images",
    "nc": 5,
    "names": [
        "Longitudinal Crack",
        "Transverse Crack",
        "Alligator Crack",
        "Other Corruption",
        "Pothole"
    ]
}

yaml_path = Path("/kaggle/working/road_damage_data.yaml") if Path("/kaggle/working").exists() else Path("road_damage_data.yaml")
with open(yaml_path, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Config saved to:", yaml_path)

In [ ]:
# STEP 3: Dataset Class Distribution & Imbalance Audit
labels_train = dataset_root / "train" / "labels"
class_counts = {i: 0 for i in range(5)}
total_boxes = 0

if labels_train.exists():
    for lf in labels_train.glob("*.txt"):
        with open(lf, "r") as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    c = int(parts[0])
                    if 0 <= c < 5:
                        class_counts[c] += 1
                        total_boxes += 1

    print("=== Dataset Distribution for Conference Paper ===")
    for i, name in enumerate(config["names"]):
        count = class_counts[i]
        pct = (count / max(1, total_boxes)) * 100
        print(f"{i}. {name}: {count} instances ({pct:.1f}%)")
else:
    print("Training labels directory not found yet. Ensure dataset is attached.")

In [ ]:
# STEP 4: YOLOv8 Training with Conference-Grade Augmentations
# Load pretrained YOLOv8 Nano/Medium model
model = YOLO("yolov8n.pt")

project_dir = "/kaggle/working/road_damage_experiments" if Path("/kaggle/working").exists() else "./runs/train"

results = model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.1,
    workers=2,
    device=0 if torch.cuda.is_available() else "cpu",
    project=project_dir,
    name="yolov8_road_damage_conference",
    pretrained=True,
    plots=True,
    save=True
)

print("Training complete! Best weights saved to:", model.trainer.best)

In [ ]:
# STEP 5: Model Validation & Conference Metrics (mAP@0.5, mAP@0.5:0.95)
metrics = model.val()
print("=== Validation Results ===")
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)
print("Class-wise AP@0.5:", metrics.box.maps)

In [ ]:
# STEP 6: Dual-Modal Inference Demo (Image & Video)
import predict_utils

# 1. Test Single Image Inference
sample_images = list(Path(dataset_root).glob("val/images/*.jpg"))
if sample_images:
    from PIL import Image
    test_img = Image.open(sample_images[0])
    annotated_img, report, json_log = predict_utils.predict_image(test_img, model, conf_thresh=0.15)
    print(report)
else:
    print("No validation image found for quick test.")